# Demux samples

refer to [`seperate_samples_helper.py`](https://github.com/cvraut/Gecko_screen2/blob/668b2726668f526915199cc473adf0d3bc12b701/code/p15_analyze_9536_data/seperate_samples_helper.py) for the core logic.

Notable changes to the demux logic. Because these are paired-end reads we actually get 2 mappings per read. This notebook will apply the simple technique to the forward read pairs first and then try to use the paired nature of the reads to be double sure of the alignment.

# Old unpaired logic

In [2]:
import pandas as pd

# s = "/home/craut/wkspce/CRISPRai_NGS/data_out/15107-PP/alignment/15107-PP-1_S1_001_pe.sam" # sam file location
s = "/home/craut/wkspce/CRISPRai_NGS/code/p05_demux_sample2/test.sam"
c = set(["6S30M115S"]) # cigar strings to screen for (the good reads)
o = "/home/craut/wkspce/CRISPRai_NGS/data_out/15107-PP/alignment/unpaired_demux_R1_only/test_demux" # header for the output files
debug=True

In [9]:
if debug:
    print(f"samfile: {s}")
    print(f"cigars: {c}")
    print(f"output_header: {o}")

header_lines = 0
with open(s) as f:
    for i,line in enumerate(f):
        if line.startswith("@"):
            header_lines += 1
        else:
            break
sam_col_names = ["QNAME","FLAG","RNAME","POS","MAPQ","CIGAR","RNEXT","PNEXT","TLEN","SEQ","QUAL","AS","XS","XN","XM","XO","XG","NM","MD","YT","ZW","ZW2","ZW3"]
if debug:
    print(f"header_lines: {header_lines}")
sam_results = pd.read_table(s,header=None,skiprows=header_lines, names=sam_col_names, index_col=False)


samfile: /home/craut/wkspce/CRISPRai_NGS/code/p05_demux_sample2/test.sam
cigars: {'6S30M115S'}
output_header: /home/craut/wkspce/CRISPRai_NGS/data_out/15107-PP/alignment/unpaired_demux_R1_only/test_demux
header_lines: 20


In [10]:
# identify the counts of the different cigar strings in the sam file
cigar_counts = sam_results["CIGAR"].value_counts()
# print the top 10 most common cigar strings and their counts if the cigar string contains "30M"
print(cigar_counts[cigar_counts.index.str.contains("30M")].head(10))

6S30M115S     19770
16S30M105S       18
4S30M117S         7
2S30M119S         4
7S30M114S         3
5S30M116S         2
3S30M118S         2
1S30M120S         2
34S30M87S         1
68S30M53S         1
Name: CIGAR, dtype: int64


In [4]:
print(sam_results.shape)

(999980, 23)


In [8]:
if debug:
    print(sam_results.shape)
unmapped = sam_results[sam_results["RNAME"] == "*"]
# write the unmapped reads to a file
with open(f"{o}_unmapped.fastq","w+") as of:
    for qname, seq, qual in zip(unmapped["QNAME"],unmapped["SEQ"],unmapped["QUAL"]):
        of.write(f"@{qname}\n{seq}\n+\n{qual}\n")
del unmapped
# filter out the reads that don't have the correct cigar string
sam_results = sam_results[sam_results["CIGAR"].isin(c)]
# sort the reads by the RNAME
sam_results.sort_values(by=["RNAME"],inplace=True)
barcode,of = None,None
for qname, rname, seq, qual in zip(sam_results["QNAME"],sam_results["RNAME"],sam_results["SEQ"],sam_results["QUAL"]):
    if barcode != rname:
        if barcode:
            of.close()
        barcode = rname
        of = open(f"{o}_{rname}.fastq","w+")
    of.write(f"@{qname}\n{seq}\n+\n{qual}\n")

(19770, 23)


In [6]:
sam_results 

,QNAME,FLAG,RNAME,POS,MAPQ,CIGAR,RNEXT,PNEXT,TLEN,SEQ,...,XN,XM,XO,XG,NM,MD,YT,ZW,ZW2,ZW3
146374,LH00346:491:22CYWMLT1:1:1101:31980:2745,99,barcode_ACAGTG,1,2,6S30M115S,=,9,165,GCGTGGACAGTGTCTTGTGGAAAGGACGAAACAACGTTGCCGTTGC...,...,XN:i:0,XM:i:1,XO:i:0,XG:i:0,NM:i:1,MD:Z:27C2,YS:i:44,YT:Z:CP,NaN,NaN
418660,LH00346:491:22CYWMLT1:1:1101:28892:5948,99,barcode_ACAGTG,1,2,6S30M115S,=,9,165,TCACGCACAGTGTCTTGTGGAAAGGACGAAACACCGGCTCGTGAGG...,...,XN:i:0,XM:i:0,XO:i:0,XG:i:0,NM:i:0,MD:Z:30,YS:i:44,YT:Z:CP,NaN,NaN
958342,LH00346:491:22CYWMLT1:1:1101:5031:12209,99,barcode_ACAGTG,1,2,6S30M115S,=,9,165,GCATTAACAGTGTCTTGTGGAAAGGACGAAACATCGGCTCCCGATA...,...,XN:i:0,XM:i:1,XO:i:0,XG:i:0,NM:i:1,MD:Z:27C2,YS:i:44,YT:Z:CP,NaN,NaN
418804,LH00346:491:22CYWMLT1:1:1101:43647:5948,99,barcode_ACAGTG,1,2,6S30M115S,=,9,165,CAAGGTACAGTGTCTTGTGGAAAGGACGAAACAACGGCGGGACAGG...,...,XN:i:0,XM:i:1,XO:i:0,XG:i:0,NM:i:1,MD:Z:27C2,YS:i:44,YT:Z:CP,NaN,NaN
419610,LH00346:491:22CYWMLT1:1:1101:22282:5964,99,barcode_ACAGTG,1,2,6S30M115S,=,9,165,TGGTAGACAGTGTCTTGTGGAAAGGACGAAACACCGTCGGCAGGTC...,...,XN:i:0,XM:i:0,XO:i:0,XG:i:0,NM:i:0,MD:Z:30,YS:i:44,YT:Z:CP,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
912316,LH00346:491:22CYWMLT1:1:1101:39358:11649,99,barcode_TTAGGC,1,2,6S30M115S,=,8,164,ATTTATTTAGGCTCTTGTGGAAAGGACGAAACACCGTGTCCCTCTC...,...,XN:i:0,XM:i:0,XO:i:0,XG:i:0,NM:i:0,MD:Z:30,YS:i:46,YT:Z:CP,NaN,NaN
804668,LH00346:491:22CYWMLT1:1:1101:25240:10384,99,barcode_TTAGGC,1,2,6S30M115S,=,9,165,AGTGTATTAGGCTCTTGTGGAAAGGACGAAACACCGCAGGCGGCGA...,...,XN:i:0,XM:i:0,XO:i:0,XG:i:0,NM:i:0,MD:Z:30,YS:i:44,YT:Z:CP,NaN,NaN
336212,LH00346:491:22CYWMLT1:1:1101:27052:4971,99,barcode_TTAGGC,1,2,6S30M115S,=,9,165,TATATTTTAGGCTCTTGTGGAAAGGACGAAACACCGTTTGTACCGA...,...,XN:i:0,XM:i:0,XO:i:0,XG:i:0,NM:i:0,MD:Z:30,YS:i:44,YT:Z:CP,NaN,NaN
664334,LH00346:491:22CYWMLT1:1:1101:34643:8766,99,barcode_TTAGGC,1,2,6S30M115S,=,9,165,ATAAGATTAGGCTCTTGTGGAAAGGACGAAACAACGGCAAGTGAGC...,...,XN:i:0,XM:i:1,XO:i:0,XG:i:0,NM:i:1,MD:Z:27C2,YS:i:44,YT:Z:CP,NaN,NaN


In [7]:
19770/999980

0.01977039540790816